# Experiment 09: Layer 0 Inactive Weights DBSCAN 3D Tensor Compression

**Target Model**: `google/gemma-3-1b-it` (Layer 0 `model.layers[0]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Innovations:
1. **Fine-Precision FP32 Activity Partitioning**:
   - Computes activation mean absolute magnitude $v_i = \frac{1}{N} \sum |x_{k, i}|$ and variance $\sigma_i^2$ in full FP32.
   - Partitions coordinates into **Active** (top 2,400) and **Inactive** (remaining 4,512 coordinates).
   - Inactive coordinates possess low dynamic range ($v_i \in [10^{-5}, 0.08]$), demanding fine-scale numerical analysis.
2. **Fine-Scale Inactive DBSCAN Clustering**:
   - Dynamically scales DBSCAN $\epsilon_{\text{inact}} = \max(10^{-4}, 0.18 \times \operatorname{std}(v_{\text{inact}}))$ in FP32 to discover natural functional groupings among low-activation neurons.
3. **Inactive Superweight & Outlier Quarantine**:
   - Preserves coordinates in uncompressed FP32 if:
     - DBSCAN label is `-1` (density noise)
     - Weight magnitude $|W_{ij}| > 3.0$
     - Variance is in top 1% of the inactive subspace
4. **Inactive 3D Tensor Formation**:
   - Groups remaining clustered inactive coordinates into $K_{\text{inact}} = 10$ uniform slices of size $M = 400$ ($4,000$ coordinates total).
   - Forms 3D Inactive Tensor: $\mathcal{T}_{\text{inact}} \in \mathbb{R}^{10 \times 400 \times 1152}$.
5. **Isolation Benchmark**:
   - Active weights ($2,400$ coords) remain in pristine FP32 to isolate the exact downstream performance effect of compressing the inactive subspace.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Ensure fast offline loading from local cache
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    print("Successfully imported ModelManagementInterface from neural_decomp")
except ImportError:
    repo_root = Path.cwd().resolve()
    while repo_root.parent != repo_root:
        if (repo_root / "neural_decomp").is_dir():
            sys.path.insert(0, str(repo_root))
            break
        repo_root = repo_root.parent
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    print("Imported ModelManagementInterface after updating sys.path")

print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


/home/dwithun/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-y0yvqond because there was an issue with the default path ({configdir}); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully imported ModelManagementInterface from neural_decomp
TensorLy Backend: pytorch
PyTorch Version: 2.14.0+cu130
CUDA Available: True
Device Name: NVIDIA GeForce RTX 3070 Ti


In [2]:
# =====================================================================
# STEP 2: Model & Dataset Loading
# =====================================================================
MODEL_ID = "google/gemma-3-1b-it"
TARGET_LAYER_IDX = 0
NUM_EVAL_SAMPLES = 150

print(f"Loading model: {MODEL_ID} on device 0...")
mmi = ModelManagementInterface(
    model_id=MODEL_ID,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

# Cache pristine weights on CPU
W_orig = {
    "gate_proj": model.model.layers[TARGET_LAYER_IDX].mlp.gate_proj.weight.data.clone().cpu(),
    "up_proj":   model.model.layers[TARGET_LAYER_IDX].mlp.up_proj.weight.data.clone().cpu(),
    "down_proj": model.model.layers[TARGET_LAYER_IDX].mlp.down_proj.weight.data.clone().cpu(),
}

print(f"Cached Layer {TARGET_LAYER_IDX} pristine weights:")
for k, v in W_orig.items():
    print(f"  {k:10s}: shape {list(v.shape)}")

# Load GLUE MNLI validation split
print(f"\nLoading GLUE MNLI dataset (150 evaluation samples)...")
dataset = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = dataset.select(range(NUM_EVAL_SAMPLES))

label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
label_tokens = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + tok, add_special_tokens=False)[0] for tok in label_tokens]
print(f"Candidate label token IDs: {list(zip(label_tokens, label_token_ids))}")


Loading model: google/gemma-3-1b-it on device 0...


Loading weights:   0%|                                 | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|                         | 1/340 [00:00<03:06,  1.81it/s]

Loading weights:  62%|█████████████▋        | 211/340 [00:00<00:00, 428.42it/s]

Loading weights:  96%|█████████████████████▏| 328/340 [00:00<00:00, 460.80it/s]

Loading weights: 100%|██████████████████████| 340/340 [00:00<00:00, 381.58it/s]

Using the latest cached version of the dataset since nyu-mll/glue couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'mnli' at /home/dwithun/.cache/huggingface/datasets/nyu-mll___glue/mnli/0.0.0/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c (last modified on Sat Aug 22 20:28:52 2026).


Cached Layer 0 pristine weights:
  gate_proj : shape [6912, 1152]
  up_proj   : shape [6912, 1152]
  down_proj : shape [1152, 6912]

Loading GLUE MNLI dataset (150 evaluation samples)...
Candidate label token IDs: [('entailment', 83155), ('neutral', 12643), ('contradiction', 38912)]
time: 2.19s
cummulative_time: 4.20s


In [3]:
# =====================================================================
# STEP 3: Tri-Hook Profiling & Pristine Baseline Accuracy
# =====================================================================
target_layer = model.model.layers[TARGET_LAYER_IDX]

acts_dict = {
    "gate_proj": [],
    "up_proj": [],
    "down_proj": [],
}

def hook_act_fn(module, input, output):
    acts_dict["gate_proj"].append(output.detach().float().cpu())

def hook_up_proj(module, input, output):
    acts_dict["up_proj"].append(output.detach().float().cpu())

def hook_down_proj(module, input, output):
    acts_dict["down_proj"].append(input[0].detach().float().cpu())

h_gate = target_layer.mlp.act_fn.register_forward_hook(hook_act_fn)
h_up = target_layer.mlp.up_proj.register_forward_hook(hook_up_proj)
h_down = target_layer.mlp.down_proj.register_forward_hook(hook_down_proj)

baseline_preds = []
ground_truths = []

model.eval()
print("Running baseline evaluation and tri-hook activation profiling...")
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Profiling Baseline"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        baseline_preds.append(pred_label)
        ground_truths.append(sample["label"])

# Remove hooks
h_gate.remove()
h_up.remove()
h_down.remove()

baseline_accuracy = accuracy_score(ground_truths, baseline_preds)
print(f"\nLayer {TARGET_LAYER_IDX} Pristine Baseline Accuracy: {baseline_accuracy * 100:.2f}%")

# Aggregate activation matrices
acts_matrix = {}
for sub_name in ["gate_proj", "up_proj", "down_proj"]:
    stacked = torch.cat(acts_dict[sub_name], dim=1).squeeze(0)  # [Total_Tokens, 6912]
    acts_matrix[sub_name] = stacked.numpy()
    print(f"Captured {sub_name} activations: {acts_matrix[sub_name].shape} (Tokens x Neurons)")


Running baseline evaluation and tri-hook activation profiling...


Profiling Baseline:   0%|                              | 0/150 [00:00<?, ?it/s]

Profiling Baseline:   1%|▏                     | 1/150 [00:00<01:19,  1.87it/s]

Profiling Baseline:   3%|▌                     | 4/150 [00:00<00:19,  7.61it/s]

Profiling Baseline:   5%|█                     | 7/150 [00:00<00:11, 12.42it/s]

...

Profiling Baseline:  94%|██████████████████▊ | 141/150 [00:05<00:00, 29.46it/s]

Profiling Baseline:  96%|███████████████████▏| 144/150 [00:05<00:00, 29.53it/s]

Profiling Baseline:  99%|███████████████████▋| 148/150 [00:05<00:00, 29.93it/s]

Profiling Baseline: 100%|████████████████████| 150/150 [00:05<00:00, 26.39it/s]


Layer 0 Pristine Baseline Accuracy: 50.67%
Captured gate_proj activations: (12936, 6912) (Tokens x Neurons)
Captured up_proj activations: (12936, 6912) (Tokens x Neurons)
Captured down_proj activations: (12936, 6912) (Tokens x Neurons)
time: 5.86s
cummulative_time: 10.06s


In [4]:
# =====================================================================
# STEP 4: FP32 Inactive Partitioning, DBSCAN & 3D Tensor Formation
# =====================================================================
NUM_ACTIVE = 2400
INACT_CHUNK_SIZE = 400
INACT_NUM_CHUNKS = 10  # 10 slices x 400 = 4000 coords

submodule_inactive_data = {}

for sub_name in ["gate_proj", "up_proj", "down_proj"]:
    is_col = (sub_name == "down_proj")
    W = W_orig[sub_name].float()
    A = acts_matrix[sub_name]
    
    # 1. Compute FP32 activity metric
    v_all = np.mean(np.abs(A), axis=0)  # [6912] in FP32
    var_all = np.var(A, axis=0)
    
    # Sort all coordinates by activity
    sorted_all = np.argsort(v_all)
    inactive_pool = sorted_all[:-NUM_ACTIVE]  # Lowest 4,512 neurons
    active_pool = sorted_all[-NUM_ACTIVE:]    # Top 2,400 neurons
    
    v_inact = v_all[inactive_pool]
    
    print(f"\n{'='*70}")
    print(f"Processing Inactive Subspace for: {sub_name}")
    print(f"  Total Inactive Neurons: {len(inactive_pool):,}")
    print(f"  Inactive FP32 Mean Range: [{v_inact.min():.6f}, {v_inact.max():.6f}], Mean={v_inact.mean():.6f}, Std={v_inact.std():.6f}")
    
    # 2. Notebook 02 Spectral Denoising (SVD 95% Energy) + 60% Sparsification Threshold
    W_sub_inact = W[:, inactive_pool].T if is_col else W[inactive_pool, :]
    U, S, Vh = torch.linalg.svd(W_sub_inact, full_matrices=False)
    cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
    r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
    W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
    eps_val = torch.quantile(torch.abs(W_denoised), 0.60)
    W_sparse = W_denoised.clone()
    W_sparse[torch.abs(W_sparse) < eps_val] = 0.0
    print(f"  SVD 95% Denoising: Rank {r95}/{len(S)} ({r95/len(S)*100:.1f}%), Threshold eps={eps_val:.5f} (60% zeros)")
    
    W_clean = W.clone()
    if is_col:
        W_clean[:, inactive_pool] = W_sparse.T
    else:
        W_clean[inactive_pool, :] = W_sparse

    # 3. Fine-precision DBSCAN on inactive activation values
    eps_inact = max(1e-4, float(np.std(v_inact) * 0.18))
    db_inact = DBSCAN(eps=eps_inact, min_samples=30, metric="euclidean")
    inact_labels = db_inact.fit_predict(v_inact.reshape(-1, 1))
    
    n_clusters = len([l for l in np.unique(inact_labels) if l != -1])
    n_noise = int(np.sum(inact_labels == -1))
    print(f"  DBSCAN on Inactive: {n_clusters} clusters, {n_noise} noise points (eps={eps_inact:.6f})")
    
    # 4. Inactive Superweight & Outlier Quarantine
    inact_mags = np.max(np.abs(A[:, inactive_pool]), axis=0)
    inact_vars = var_all[inactive_pool]
    super_mask = (inact_labels == -1) | (inact_mags >= np.quantile(inact_mags, 0.99)) | (inact_vars >= np.quantile(inact_vars, 0.99))
    
    super_coords = inactive_pool[super_mask]
    clustered_coords = inactive_pool[~super_mask]
    print(f"  Inactive Superweights Quarantined: {len(super_coords):,}")
    print(f"  Clean Clustered Inactive Coords:   {len(clustered_coords):,}")
    
    # 5. Form uniform chunks of size 400
    chunk_list = []
    unique_labs = [l for l in np.unique(inact_labels) if l != -1]
    
    for lab in unique_labs:
        c_sub_idx = np.where((inact_labels == lab) & (~super_mask))[0]
        if len(c_sub_idx) == 0:
            continue
        c_coords = inactive_pool[c_sub_idx]
        sorted_c = c_coords[np.argsort(v_all[c_coords])]
        num_full = len(sorted_c) // INACT_CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_c[ci * INACT_CHUNK_SIZE : (ci + 1) * INACT_CHUNK_SIZE])
            if len(chunk_list) >= INACT_NUM_CHUNKS:
                break
        if len(chunk_list) >= INACT_NUM_CHUNKS:
            break
            
    # Fill any remaining chunks from clustered pool
    if len(chunk_list) < INACT_NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [c for c in clustered_coords if c not in assigned]
        needed = INACT_NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= INACT_CHUNK_SIZE:
                chunk_list.append(np.array(avail[:INACT_CHUNK_SIZE]))
                avail = avail[INACT_CHUNK_SIZE:]
                
    active_inact_coords = np.concatenate(chunk_list)
    print(f"  Created {len(chunk_list)} inactive chunks ({len(active_inact_coords):,} coordinates)")
    
    # 6. Form 3D Inactive Tensor [10, 400, 1152] from Denoised & Thresholded Weights
    if is_col:
        T_inact = torch.stack([W_clean[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T_inact = torch.stack([W_clean[c, :].float().cpu() for c in chunk_list], dim=0)
        
    print(f"  Inactive 3D Tensor Shape: {list(T_inact.shape)} ({T_inact.numel():,} parameters)")
    
    submodule_inactive_data[sub_name] = {
        "tensor": T_inact,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "active_pool": active_pool,  # Untouched pristine active coords
        "is_col": is_col,
    }



Processing Inactive Subspace for: gate_proj
  Total Inactive Neurons: 4,512
  Inactive FP32 Mean Range: [0.009059, 0.184104], Mean=0.104886, Std=0.043164


  SVD 95% Denoising: Rank 959/1152 (83.2%), Threshold eps=0.02520 (60% zeros)
  DBSCAN on Inactive: 1 clusters, 0 noise points (eps=0.007769)


  Inactive Superweights Quarantined: 88
  Clean Clustered Inactive Coords:   4,424
  Created 10 inactive chunks (4,000 coordinates)
  Inactive 3D Tensor Shape: [10, 400, 1152] (4,608,000 parameters)



Processing Inactive Subspace for: up_proj
  Total Inactive Neurons: 4,512
  Inactive FP32 Mean Range: [0.358382, 0.785526], Mean=0.663025, Std=0.080111


  SVD 95% Denoising: Rank 947/1152 (82.2%), Threshold eps=0.02335 (60% zeros)
  DBSCAN on Inactive: 1 clusters, 22 noise points (eps=0.014420)


  Inactive Superweights Quarantined: 102
  Clean Clustered Inactive Coords:   4,410
  Created 10 inactive chunks (4,000 coordinates)
  Inactive 3D Tensor Shape: [10, 400, 1152] (4,608,000 parameters)



Processing Inactive Subspace for: down_proj
  Total Inactive Neurons: 4,512
  Inactive FP32 Mean Range: [0.009038, 0.133699], Mean=0.079232, Std=0.030030


  SVD 95% Denoising: Rank 921/1152 (79.9%), Threshold eps=0.00994 (60% zeros)
  DBSCAN on Inactive: 1 clusters, 0 noise points (eps=0.005405)


  Inactive Superweights Quarantined: 58
  Clean Clustered Inactive Coords:   4,454
  Created 10 inactive chunks (4,000 coordinates)
  Inactive 3D Tensor Shape: [10, 400, 1152] (4,608,000 parameters)
time: 3.40s
cummulative_time: 13.47s


In [5]:
# =====================================================================
# STEP 5: Define Tucker GD Optimizer & Inactive Evaluation Tiers
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device="cpu"):
    safe_ranks = [
        min(ranks[0], T.shape[0] - 1),
        min(ranks[1], T.shape[1] - 1),
        min(ranks[2], T.shape[2] - 1),
    ]
    core_init, factors_init = tucker(T, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

# Multi-tier inactive rank sweeps
inactive_eval_tiers = [
    {
        "name": "Inactive Conservative",
        "ranks": [6, 160, 500],
        "description": "High fidelity on inactive subspace (~70% error)"
    },
    {
        "name": "Inactive Moderate (Sweet Spot)",
        "ranks": [5, 120, 350],
        "description": "Calibrated moderate compression on inactive subspace (~77% error)"
    },
    {
        "name": "Inactive Aggressive",
        "ranks": [4, 80, 200],
        "description": "High parameter cut on inactive subspace (~85% error)"
    },
    {
        "name": "Inactive Ultra-Aggressive",
        "ranks": [3, 40, 100],
        "description": "Extreme parameter cut on inactive subspace (~90% error)"
    },
]

print("Defined Tucker GD optimizer and 4 inactive evaluation tiers.")


Defined Tucker GD optimizer and 4 inactive evaluation tiers.
time: 0.00s
cummulative_time: 13.48s


In [6]:
# =====================================================================
# STEP 6: Execute Inactive Compression Sweeps & Evaluate
# =====================================================================
benchmarks = []

for tier in inactive_eval_tiers:
    tier_name = tier["name"]
    ranks = tier["ranks"]
    
    print(f"\n{'='*80}")
    print(f"Running Evaluation for Tier: {tier_name} (Ranks: {ranks})")
    print(f"{'='*80}")
    
    tier_errors = {}
    tier_params_saved = 0
    
    # 1. Factorize inactive tensors and inject into live model
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = submodule_inactive_data[sub_name]
        T_inact = sdata["tensor"]
        
        cg, fg, T_recon, err = optimize_tucker_gd(
            T_inact, ranks=ranks, num_steps=35, lr=1e-3, device="cpu"
        )
        tier_errors[sub_name] = err
        
        orig_p = T_inact.numel()
        comp_p = cg.numel() + sum(f.numel() for f in fg)
        tier_params_saved += (orig_p - comp_p)
        
        # Inject into live layer
        mod = getattr(target_layer.mlp, sub_name)
        mod.weight.data = W_orig[sub_name].clone().to(model.device)
        
        if sdata["is_col"]:
            for k, c in enumerate(sdata["chunk_list"]):
                mod.weight.data[:, c] = T_recon[k].T.to(device=model.device, dtype=mod.weight.dtype)
            if len(sdata["super_coords"]) > 0:
                mod.weight.data[:, sdata["super_coords"]] = W_orig[sub_name][:, sdata["super_coords"]].to(model.device)
        else:
            for k, c in enumerate(sdata["chunk_list"]):
                mod.weight.data[c, :] = T_recon[k].to(device=model.device, dtype=mod.weight.dtype)
            # Re-ensure superweights in exact FP32
            if len(sdata["super_coords"]) > 0:
                mod.weight.data[sdata["super_coords"], :] = W_orig[sub_name][sdata["super_coords"], :].to(model.device)
            
        print(f"  {sub_name:10s}: Recon Err = {err*100:.2f}%, Params Saved = {orig_p - comp_p:,}")
        
    print(f"Total Inactive Parameters Eliminated (Layer 0): {tier_params_saved:,}")
    
    # 2. Evaluate GLUE MNLI
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy
    
    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Inactive Params Cut: {tier_params_saved:,}")
    
    benchmarks.append({
        "Variant": tier_name,
        "Ranks": ranks,
        "Gate_Err": round(tier_errors["gate_proj"] * 100, 2),
        "Up_Err": round(tier_errors["up_proj"] * 100, 2),
        "Down_Err": round(tier_errors["down_proj"] * 100, 2),
        "Params_Cut": tier_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore Layer 0 pristine weights
for sub_name in ["gate_proj", "up_proj", "down_proj"]:
    getattr(target_layer.mlp, sub_name).weight.data = W_orig[sub_name].clone().to(model.device)
print("\nRestored Layer 0 to pristine weights.")



Running Evaluation for Tier: Inactive Conservative (Ranks: [6, 160, 500])


  gate_proj : Recon Err = 80.82%, Params Saved = 3,487,940


  up_proj   : Recon Err = 83.27%, Params Saved = 3,487,940


  down_proj : Recon Err = 81.87%, Params Saved = 3,487,940
Total Inactive Parameters Eliminated (Layer 0): 10,463,820


Evaluating Inactive Conservative:   0%|                | 0/150 [00:00<?, ?it/s]

...

Evaluating Inactive Ultra-Aggressive:  93%|█▊| 140/150 [00:04<00:00, 34.99it/s]

Evaluating Inactive Ultra-Aggressive:  96%|█▉| 144/150 [00:04<00:00, 34.90it/s]

Evaluating Inactive Ultra-Aggressive:  99%|█▉| 148/150 [00:04<00:00, 34.18it/s]

Evaluating Inactive Ultra-Aggressive: 100%|██| 150/150 [00:04<00:00, 33.94it/s]


Result for Inactive Ultra-Aggressive:
  Downstream Accuracy: 47.33% (Δ vs Baseline: -3.33%)
  Inactive Params Cut: 13,394,310

Restored Layer 0 to pristine weights.
time: 51.31s
cummulative_time: 64.84s


In [7]:
# =====================================================================
# STEP 7: Benchmark Summary & JSON Artifact Export
# =====================================================================
print(f"\n{'='*105}")
print(f"{'Variant':<28} | {'Ranks':<16} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<11} | {'Accuracy':<9} | {'Delta':<8}")
print(f"{'='*105}")
print(f"{'Baseline (Uncompressed)':<28} | {'Full':<16} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<11} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for b in benchmarks:
    ranks_str = str(b["Ranks"])
    print(f"{b['Variant']:<28} | {ranks_str:<16} | {b['Gate_Err']:>6.2f}%  | {b['Up_Err']:>5.2f}%  | {b['Down_Err']:>6.2f}%   | {b['Params_Cut']:<11,d} | {b['Accuracy']:>7.2f}% | {b['Delta']:>+6.2f}%")
print(f"{'='*105}")

# Save JSON artifact
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "09_inactive_dbscan_tucker_results.json"

payload = {
    "model_id": MODEL_ID,
    "layer": TARGET_LAYER_IDX,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "benchmarks": benchmarks,
}

with open(results_file, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nSaved benchmark results to {results_file}")



Variant                      | Ranks            | Gate Err  | Up Err   | Down Err  | Params Cut  | Accuracy  | Delta   
Baseline (Uncompressed)      | Full             | 0.00%     | 0.00%    | 0.00%     | 0           |   50.67% | +0.00%  
Inactive Conservative        | [6, 160, 500]    |  80.82%  | 83.27%  |  81.87%   | 10,463,820  |   53.33% |  +2.67%
Inactive Moderate (Sweet Spot) | [5, 120, 350]    |  86.71%  | 89.22%  |  87.91%   | 11,840,250  |   46.67% |  -4.00%
Inactive Aggressive          | [4, 80, 200]     |  91.58%  | 93.96%  |  92.80%   | 12,844,680  |   49.33% |  -1.33%
Inactive Ultra-Aggressive    | [3, 40, 100]     |  95.27%  | 97.22%  |  96.29%   | 13,394,310  |   47.33% |  -3.33%

Saved benchmark results to artifacts/09_inactive_dbscan_tucker_results.json
time: 0.00s
cummulative_time: 64.85s
